# R09 EEG–握力特征池展示

本笔记本从团队数据接口读取 R09，使用 `literature_all` 配方生成因果 EEG 特征池，并展示特征、标签、时间窗和通道分布。

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from grip_data_interface import load_flight_trials
from grip_feature_pool import build_grip_feature_pool

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
})
BLUE = '#2F6B9A'
GOLD = '#C6922F'
INK = '#252A30'
GRID = '#D9DEE3'

DATA_DIR = Path('0807华山grip flight')
EEG_PATH = DATA_DIR / 'testS001R09.dat.larkcache'
TASK_PATH = DATA_DIR / 'testS001R09_1.dat'
RECIPE = 'literature_all'
WINDOW_MS = 500
STEP_MS = 50

assert EEG_PATH.exists(), EEG_PATH
assert TASK_PATH.exists(), TASK_PATH

In [ ]:
started = perf_counter()
r09_trials = load_flight_trials(
    EEG_PATH,
    TASK_PATH,
    segment='trial',
)
pool = build_grip_feature_pool(
    r09_trials,
    recipe=RECIPE,
    window_ms=WINDOW_MS,
    step_ms=STEP_MS,
    causal=True,
    reference='none',
)
elapsed_s = perf_counter() - started
pool.validate()

## tl;dr

In [ ]:
n_windows, n_channels, n_features = pool.X.shape
flight_windows = int(pool.windows['mask_flight'].sum())
summary = pd.DataFrame({
    'item': ['recording', 'trials', 'windows', 'flight windows', 'channels', 'features', 'feature array', 'build time'],
    'value': [
        pool.manifest['recording_ids'][0],
        len(r09_trials['meta']),
        f'{n_windows:,}',
        f'{flight_windows:,}',
        n_channels,
        n_features,
        f'{n_windows} × {n_channels} × {n_features}',
        f'{elapsed_s:.1f} s',
    ],
})
display(summary.style.hide(axis='index'))
display(Markdown(
    f'R09 共生成 **{n_windows:,}** 个历史窗口，每个窗口包含 **{n_channels}** 个通道和 '
    f'**{n_features}** 个文献特征；其中 **{flight_windows:,}** 个窗口落在 flight 阶段。'
))

## Context & Methods

### Key assumptions

- 上游使用 `segment='trial'`，使 flight 开始处的窗口保留 Countdown 历史。
- 每个特征窗口只使用标签时刻及之前的 EEG；滤波使用因果模式。
- 频带名称复现文献定义，但这不是对每篇论文全部预处理细节的逐项复现。
- GamePhase、Collision 和 outcome 仅作为标签或筛选条件，不进入 EEG 特征。

In [ ]:
method_summary = pd.Series({
    'source EEG': EEG_PATH.name,
    'source task': TASK_PATH.name,
    'sampling rate (Hz)': pool.manifest['sampling_rate_hz'],
    'recipe': pool.manifest['recipe'],
    'window / step (ms)': f'{WINDOW_MS} / {STEP_MS}',
    'causal filtering': pool.manifest['causal'],
    'reference': pool.manifest['reference'],
    'notches (Hz)': ', '.join(map(str, pool.manifest['notch_hz'])),
    'band-power unit': 'dB(source signal units²)',
}, name='value').to_frame()
display(method_summary)

## Data

### Trial 与特征目录

In [ ]:
trial_columns = ['trial_key', 'outcome', 'collision', 'duration_s', 'n_samples', 'force_mean', 'force_std', 'force_max']
display(r09_trials['meta'][trial_columns].style.format({
    'duration_s': '{:.2f}', 'force_mean': '{:.4f}', 'force_std': '{:.4f}', 'force_max': '{:.4f}'
}))

feature_catalog = pd.DataFrame(pool.feature_info['feature_axis'])
feature_catalog['sources'] = feature_catalog['recipe_sources'].apply(', '.join)
display(feature_catalog[['index', 'name', 'family', 'low_hz', 'high_hz', 'unit', 'sources']].fillna('—'))

### 窗口与标签预览

`window_id` 是 `pool.X` 第一维的行号；每一行都能追溯到 recording、trial 和标签时刻。

In [ ]:
preview_columns = [
    'window_id', 'trial_key', 'window_start_s', 'window_end_s', 'label_time_s',
    'game_phase_label', 'mask_flight', 'force_normalized', 'force_derivative',
    'collision_onset', 'outcome',
]
window_preview = pool.windows.merge(
    pool.labels.drop(columns=['outcome', 'trial_collision']),
    on='window_id',
    validate='one_to_one',
)
display(window_preview[preview_columns].head(10).style.format({
    'window_start_s': '{:.3f}', 'window_end_s': '{:.3f}', 'label_time_s': '{:.3f}',
    'force_normalized': '{:.4f}', 'force_derivative': '{:.4f}',
}))

## Results

### 1. 全部文献特征的时间概览

先对通道取中位数，再对每个特征沿时间做 robust z-score。颜色表示相对变化，不比较不同频带的绝对 dB 数值。

In [ ]:
def robust_z(values, axis=0):
    values = np.asarray(values, dtype=float)
    median = np.nanmedian(values, axis=axis, keepdims=True)
    mad = np.nanmedian(np.abs(values - median), axis=axis, keepdims=True)
    scale = np.where(mad > 0, 1.4826 * mad, np.nanstd(values, axis=axis, keepdims=True))
    return (values - median) / np.where(scale > 0, scale, 1.0)

time_feature = np.nanmedian(pool.X, axis=1)
time_feature_z = np.clip(robust_z(time_feature, axis=0), -3, 3)
trial_boundaries = np.flatnonzero(pool.windows['trial_key'].to_numpy()[1:] != pool.windows['trial_key'].to_numpy()[:-1]) + 1

fig, ax = plt.subplots(figsize=(15, 7))
image = ax.imshow(time_feature_z.T, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=-3, vmax=3)
for boundary in trial_boundaries:
    ax.axvline(boundary - 0.5, color=INK, lw=0.8, alpha=0.7)
ax.set_yticks(np.arange(n_features))
ax.set_yticklabels(pool.feature_names, fontsize=8)
ax.set_xlabel('Sequential feature windows (trial boundaries in black)')
ax.set_ylabel('Feature')
ax.set_title('R09 literature feature pool over time')
colorbar = fig.colorbar(image, ax=ax, pad=0.01)
colorbar.set_label('Robust z-score across windows')
fig.tight_layout()
plt.show()

### 2. 通道—特征分布

对 flight 窗口取时间中位数，再在每个特征内跨通道标准化，用于定位相对高/低功率通道。

In [ ]:
flight_mask = pool.windows['mask_flight'].to_numpy(dtype=bool)
channel_feature = np.nanmedian(pool.X[flight_mask], axis=0)
channel_feature_z = np.clip(robust_z(channel_feature, axis=0), -3, 3)

fig, ax = plt.subplots(figsize=(15, 10))
image = ax.imshow(channel_feature_z, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=-3, vmax=3)
ax.set_xticks(np.arange(n_features))
ax.set_xticklabels(pool.feature_names, rotation=60, ha='right', fontsize=8)
channel_tick_step = max(1, n_channels // 20)
channel_ticks = np.arange(0, n_channels, channel_tick_step)
ax.set_yticks(channel_ticks)
ax.set_yticklabels(np.asarray(pool.channel_names)[channel_ticks], fontsize=8)
ax.set_xlabel('Feature')
ax.set_ylabel('EEG channel')
ax.set_title('R09 channel × feature profile during flight')
colorbar = fig.colorbar(image, ax=ax, pad=0.01)
colorbar.set_label('Robust z-score across channels')
fig.tight_layout()
plt.show()

### 3. 单个 Trial：特征与握力时间关系

展示 Trial 1 的跨通道中位特征。各曲线分别标准化，因此只比较时间形态，不比较原始幅值。

In [ ]:
selected_features = [
    'lmp', 'bandpower_7_20Hz', 'bandpower_70_115Hz',
    'bandpower_130_200Hz', 'bandpower_200_300Hz',
]
first_trial = pool.windows['trial_key'].iloc[0]
trial_mask = (pool.windows['trial_key'] == first_trial).to_numpy()
trial_time = pool.windows.loc[trial_mask, 'label_time_s'].to_numpy()
trial_force = pool.labels.loc[trial_mask, 'force_normalized'].to_numpy()
trial_feature = np.nanmedian(pool.X[trial_mask], axis=1)
force_scale = np.nanstd(trial_force)
force_z = np.clip(
    (trial_force - np.nanmean(trial_force)) / (force_scale if force_scale > 0 else 1.0),
    -4, 4,
)

fig, axes = plt.subplots(len(selected_features), 1, figsize=(13, 11), sharex=True)
for ax, feature_name in zip(axes, selected_features):
    feature_index = pool.feature_names.index(feature_name)
    feature_z = np.clip(robust_z(trial_feature[:, feature_index], axis=0), -4, 4)
    ax.plot(trial_time, force_z, color=INK, lw=1.5, label='Force (z)')
    ax.plot(trial_time, feature_z, color=BLUE, lw=1.0, alpha=0.85, label=feature_name)
    ax.axhline(0, color=GRID, lw=0.8)
    ax.set_ylabel('z')
    ax.set_title(feature_name, loc='left', fontsize=10)
axes[0].legend(frameon=False, ncol=2, loc='upper right')
axes[-1].set_xlabel('Trial-relative label time (s)')
fig.suptitle(f'{first_trial}: force and channel-median EEG features', y=1.01, fontweight='bold')
fig.tight_layout()
plt.show()

### 4. High-gamma 与握力相关性最高的通道

这是描述性筛查，不是交叉验证后的建模结果。通道排名不能直接当作可泛化的通道选择。

In [ ]:
high_gamma_name = 'bandpower_70_115Hz'
high_gamma_index = pool.feature_names.index(high_gamma_name)
flight_force = pool.labels.loc[flight_mask, 'force_normalized'].to_numpy()
channel_correlations = np.asarray([
    pd.Series(pool.X[flight_mask, channel_i, high_gamma_index]).corr(
        pd.Series(flight_force), method='spearman'
    )
    for channel_i in range(n_channels)
])
top_indices = np.argsort(np.nan_to_num(np.abs(channel_correlations), nan=-1))[-15:]
top_table = pd.DataFrame({
    'channel': np.asarray(pool.channel_names)[top_indices],
    'spearman_r': channel_correlations[top_indices],
}).sort_values('spearman_r')

fig, ax = plt.subplots(figsize=(8, 6))
colors = [GOLD if value >= 0 else BLUE for value in top_table['spearman_r']]
ax.barh(top_table['channel'], top_table['spearman_r'], color=colors, edgecolor=INK, linewidth=0.4)
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel('Spearman r with normalized force')
ax.set_ylabel('EEG channel')
ax.set_title(f'R09 top |correlation| channels: {high_gamma_name}')
ax.grid(axis='x', color=GRID, linewidth=0.6)
fig.tight_layout()
plt.show()
display(top_table.sort_values('spearman_r', ascending=False).reset_index(drop=True).style.format({'spearman_r': '{:.3f}'}))

## Checks

检查特征、标签、窗口索引是否一一对应，并演示下游接口。

In [ ]:
checks = {
    'X / labels / windows row alignment': len(pool.labels) == len(pool.windows) == len(pool.X),
    'feature values finite': bool(np.isfinite(pool.X).all()),
    'window_id sequential': bool(np.array_equal(pool.windows['window_id'], np.arange(len(pool.windows)))),
    'channel metadata aligned': len(pool.channel_names) == pool.X.shape[1],
    'feature metadata aligned': len(pool.feature_names) == pool.X.shape[2],
    'all trials represented': pool.windows['trial_key'].nunique() == len(r09_trials['meta']),
}
check_table = pd.DataFrame({'check': checks.keys(), 'passed': checks.values()})
display(check_table.style.hide(axis='index'))
assert all(checks.values())

In [ ]:
X, y, groups = pool.as_sklearn(
    target='force_normalized',
    mask='mask_flight',
)
downstream_summary = pd.Series({
    'X shape': str(X.shape),
    'y shape': str(y.shape),
    'unique trial groups': len(np.unique(groups)),
    'group examples': ', '.join(map(str, np.unique(groups)[:3])),
}, name='value').to_frame()
display(downstream_summary)

## Takeaways

- `pool.X` 保留 `window × channel × feature` 结构，适合通道和频段消融。
- `pool.labels` 保存连续握力、握力变化率、GamePhase、Collision onset 和 outcome。
- `pool.windows` 负责把每一行追溯到 recording、trial 和时间，并提供 flight/playing mask。
- 建模时使用 `groups=trial_key` 做完整 trial 划分；不要随机拆分重叠窗口。
- high-gamma 通道相关图仅用于描述，不应替代训练集内的通道选择。
- 每个 trial 开头的低频特征可能包含因果滤波 settling；正式建模可增加 warm-up mask 或提供更长的 trial 前历史。

## Optional export

安装 `pyarrow` 后可取消下面代码的注释，写出五文件特征数据集。生成目录属于患者派生数据，不应提交到 Git。

In [ ]:
# pool.save('generated_features/R09_literature_all_v1')